# دفترچهٔ فصل ۵: تمرکز نمونه‌برداری‌شده و فاز هندسی

**شناسه‌ها:** `ms-08-metalenses` و `ms-06-polarization`. **دو مدل صریحاً جدا** داریم: طیف زاویه‌ای اسکالر همگن برای کانون خطی استوانه‌ای و جونز تحلیلی عنصر ناهمسانگرد ایده‌آل. میدان دهانهٔ کتابخانه‌ای از ضرایب تناوبی فصل ۵ تقریب موضعی می‌گیرد؛ فلزنس متناهی تمام‌موج نیست. جونز، شبیه‌سازی قطبش ریج TE نیست. بستهٔ کامل، وابستگی‌ها را حفظ می‌کند.


In [ ]:
from pathlib import Path
from dataclasses import replace,asdict
import sys,json,csv,hashlib,platform,html
import numpy as np
import scipy,matplotlib
import matplotlib.pyplot as plt
from IPython.display import display,HTML
ROOT=next((p for p in [Path.cwd(),*list(Path.cwd().parents)[:3]] if (p/'simulations/metasurface_synthesis.py').is_file()),None)
if ROOT is None:raise FileNotFoundError('Extract the Chapter 5 source bundle, preserving simulations/, data/, and notebooks/.')
sys.path.insert(0,str(ROOT/'simulations'))
from unit_cell_te import Cell,solve_cell,sample_ey,phase_error_deg
from metasurface_synthesis import solve_supercell,staircase_coefficients,select_widths,angular_spectrum_1d,rotated_jones,line_fwhm,SOLVER_VERSION
DATA=ROOT/'data/chapter-05';DATA.mkdir(parents=True,exist_ok=True)
def table(headers,rows):
    def escape(x):return html.escape(f'{x:.7g}' if isinstance(x,(float,np.floating)) else str(x))
    display(HTML('<table><thead><tr>'+''.join('<th>'+escape(x)+'</th>' for x in headers)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+escape(x)+'</td>' for x in row)+'</tr>' for row in rows)+'</tbody></table>'))
plt.rcParams.update({'figure.dpi':125,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
print('Python',platform.python_version(),'NumPy',np.__version__,'SciPy',scipy.__version__,'Matplotlib',matplotlib.__version__)

# Editable scalar aperture parameters.
WAVELENGTH_UM=1.55;OUTPUT_INDEX=1.5;FOCAL_DISTANCE_UM=30.
PITCH_UM=.65;N_APERTURE_CELLS=48;APERTURE_UM=PITCH_UM*N_APERTURE_CELLS
DX_UM=.10;N_FFT=2048;FOCUS_WINDOW_HALF_UM=1.
data=json.loads((DATA/'beam-steering.json').read_text())
rows=data['extended_library']
print('Aperture width',APERTURE_UM,'um; nominal line-focus distance',FOCAL_DISTANCE_UM,'um')

## ۱. فاز عدسی استوانه‌ای را استخراج و نمونه‌برداری کنید

در تابش عمودی، فاز تجمعی در $(x,z)=(0,f)$ با $\phi(x)=-k_t[\sqrt{x^2+f^2}-f]$ یکسان می‌شود. حد پیرامحوری $-k_tx^2/(2f)$ است؛ اینجا فاز اسکالر دقیق را می‌گیریم. دهانه یک‌بعدی و کانون خطی در $y$ است. میدان پیوسته، فاز نمونه‌شده در هر گام و میدان کتابخانه‌ای را بسازید. شبکهٔ ثابت آفست، RMS انتخاب را کمینه می‌کند؛ هیچ سنجهٔ کانون منتشرشده در انتخاب دخالت ندارد.


In [ ]:
k=2*np.pi*OUTPUT_INDEX/WAVELENGTH_UM
cell_centers=(np.arange(N_APERTURE_CELLS)+.5)*PITCH_UM-APERTURE_UM/2
phase_cells=-k*(np.sqrt(cell_centers**2+FOCAL_DISTANCE_UM**2)-FOCAL_DISTANCE_UM)
choices=[]
for offset in np.linspace(-np.pi,np.pi,361,endpoint=False):
    selected,error=select_widths(rows,phase_cells,.8,float(offset))
    choices.append((float(np.mean(error**2)),float(offset),selected))
score,phase_offset,selected=min(choices,key=lambda v:v[0])
local_t=np.array([complex(r['t_real'],r['t_imag']) for r in selected])
def aperture_fields(dx,n):
    x=(np.arange(n)-n//2)*dx
    gate=abs(x)<APERTURE_UM/2
    bins=np.clip(np.floor((x+APERTURE_UM/2)/PITCH_UM).astype(int),0,N_APERTURE_CELLS-1)
    continuous=np.where(gate,np.exp(-1j*k*(np.sqrt(x*x+FOCAL_DISTANCE_UM**2)-FOCAL_DISTANCE_UM)),0)
    sampled=np.where(gate,np.exp(1j*phase_cells[bins]),0)
    local=np.where(gate,local_t[bins],0)
    return x,dict(continuous=continuous,sampled=sampled,local_library=local)
x,apertures=aperture_fields(DX_UM,N_FFT)
focus={name:angular_spectrum_1d(field,DX_UM,WAVELENGTH_UM,FOCAL_DISTANCE_UM,OUTPUT_INDEX) for name,field in apertures.items()}
ideal_peak=float(np.max(abs(focus['continuous'])**2))
metrics=[]
for name,field in focus.items():
    intensity=abs(field)**2;roi=abs(x)<=FOCUS_WINDOW_HALF_UM
    metrics.append(dict(model=name,fwhm_um=line_fwhm(x,intensity),relative_peak=float(intensity.max()/ideal_peak),line_intensity_fraction=float(intensity[roi].sum()/intensity.sum())))
table(['Scalar aperture model','FWHM (um)','Peak / ideal peak','Line-intensity fraction within +/-1 um'],[[r[k] for k in ['model','fwhm_um','relative_peak','line_intensity_fraction']] for r in metrics])
fig,axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
for name,field in apertures.items():axes[0].plot(x,np.angle(field),label=name)
axes[0].set(xlim=(-APERTURE_UM/2,APERTURE_UM/2),xlabel='x (um)',ylabel='Wrapped aperture phase (rad)',title='Continuous, sampled, and library fields')
for name,field in focus.items():axes[1].plot(x,abs(field)**2/ideal_peak,label=name)
axes[1].set(xlim=(-4,4),xlabel='x at z=f (um)',ylabel='Squared field / ideal peak',title='Scalar line-focus profile')
for ax in axes:ax.grid(alpha=.2);ax.legend()
plt.show()

## ۲. کنترل انتشار اسکالر

FFT پنجرهٔ تناوبی فرض می‌کند. اندازهٔ پنجره و فاصلهٔ نمونه را مستقل تغییر و ناحیهٔ کانون را بررسی کنید. کنترل‌ها مربوط به **مدل انتشار اسکالر** هستند، نه همگرایی فلزنس فیزیکی. موج تخت همگن نمونه‌شده فاز انتشار دقیق دارد و بین‌های فوریهٔ منتشرشونده نرم میدان گسسته را حفظ می‌کنند.


In [ ]:
baseline=abs(focus['continuous'])**2
refinements=[]
for dx,n in [(.10,1024),(.10,4096),(.05,4096)]:
    xr,fields=aperture_fields(dx,n)
    propagated=angular_spectrum_1d(fields['continuous'],dx,WAVELENGTH_UM,FOCAL_DISTANCE_UM,OUTPUT_INDEX)
    intensity=abs(propagated)**2
    roi=abs(x)<4
    error=float(np.max(abs(np.interp(x[roi],xr,intensity)-baseline[roi]))/ideal_peak)
    refinements.append(dict(dx_um=dx,window_um=dx*n,fwhm_um=line_fwhm(xr,intensity),roi_peak_normalized_difference=error))
table(['dx (um)','FFT window (um)','FWHM (um)','Max focal-region difference / reference peak'],[[r[k] for k in ['dx_um','window_um','fwhm_um','roi_peak_normalized_difference']] for r in refinements])
assert refinements[1]['roi_peak_normalized_difference']<.02
assert refinements[2]['roi_peak_normalized_difference']<.03
mode=3;plane=np.exp(2j*np.pi*mode*np.arange(N_FFT)/N_FFT)
kx_mode=2*np.pi*mode/(N_FFT*DX_UM)
exact=plane*np.exp(1j*np.sqrt(k*k-kx_mode*kx_mode)*FOCAL_DISTANCE_UM)
propagated=angular_spectrum_1d(plane,DX_UM,WAVELENGTH_UM,FOCAL_DISTANCE_UM,OUTPUT_INDEX)
assert np.max(abs(propagated-exact))<1e-10
spectrum=np.zeros(N_FFT,complex);spectrum[[0,2,7,-3]]=[1,2j,.4,1-.2j]
test=np.fft.ifft(spectrum);after=angular_spectrum_1d(test,DX_UM,WAVELENGTH_UM,FOCAL_DISTANCE_UM,OUTPUT_INDEX)
assert abs(np.linalg.norm(after)-np.linalg.norm(test))<1e-12
z_scan=np.linspace(20,40,81)
axial=np.array([abs(angular_spectrum_1d(apertures['continuous'],DX_UM,WAVELENGTH_UM,float(z),OUTPUT_INDEX)[N_FFT//2])**2 for z in z_scan])
print('Nominal focus:',FOCAL_DISTANCE_UM,'um; sampled on-axis maximum:',z_scan[np.argmax(axial)],'um (finite-aperture focal shift is possible).')
fig,ax=plt.subplots(figsize=(7,3.4),layout='constrained');ax.plot(z_scan,axial/ideal_peak);ax.axvline(FOCAL_DISTANCE_UM,color='gray',ls='--');ax.set(xlabel='z (um)',ylabel='On-axis squared field / ideal peak at f',title='Scalar finite-aperture axial response');ax.grid(alpha=.2);plt.show()

## ۳. جونز و فاز هندسی

در پایهٔ ثابت آزمایشگاه $\mathbf e_\sigma=(1,i\sigma)^T/\sqrt2$ با $\sigma=\pm1$ بگیرید:

$$
J(\theta)=R(\theta)\,\mathrm{diag}(t_x,t_y)R(-\theta).
$$

ضریب همان پایه $(t_x+t_y)/2$ و تبدیل به $\mathbf e_{-\sigma}$ برابر $(t_x-t_y)e^{2i\sigma\theta}/2$ است. نیم‌موج $t_x=1,t_y=-1$ تبدیل کامل با فاز $2\sigma\theta$ می‌دهد.

برای تأخیرانداز بی‌اتلاف $t_x=e^{i\delta/2},t_y=e^{-i\delta/2}$، کسر تبدیل $\sin^2(\delta/2)$ است. زاویهٔ چرخش دامنهٔ تبدیل بزرگ را تضمین نمی‌کند. جونز تحلیلی محیط مشترک دارد و پاسخ برداری ریج TE نیست.


In [ ]:
angles=np.linspace(0,np.pi,181);retardances=np.linspace(0,np.pi,181)
phase_curves=[];max_jones_error=0.
for sigma in [1,-1]:
    incident=np.array([1,1j*sigma])/np.sqrt(2);converted=np.array([1,-1j*sigma])/np.sqrt(2)
    amplitude=[]
    for theta in angles:
        J=rotated_jones(1.,-1.,float(theta));a=np.vdot(converted,J@incident)
        max_jones_error=max(max_jones_error,float(abs(a-np.exp(2j*sigma*theta))),float(np.linalg.norm(J.conj().T@J-np.eye(2))))
        amplitude.append(a)
    phase_curves.append(np.unwrap(np.angle(amplitude)))
converted_power=[]
incident=np.array([1,1j])/np.sqrt(2);opposite=np.array([1,-1j])/np.sqrt(2)
for delta in retardances:
    J=rotated_jones(np.exp(.5j*delta),np.exp(-.5j*delta),.37)
    converted_power.append(abs(np.vdot(opposite,J@incident))**2)
assert max_jones_error<1e-12
assert max(abs(np.array(converted_power)-np.sin(retardances/2)**2))<1e-12
fig,axes=plt.subplots(1,2,figsize=(10,3.6),layout='constrained')
for sigma,phase in zip([1,-1],phase_curves):axes[0].plot(np.rad2deg(angles),np.rad2deg(phase),label=f'sigma={sigma:+d}')
axes[0].set(xlabel='Element rotation (deg)',ylabel='Unwrapped converted phase (deg)',title='Ideal half-wave geometric phase');axes[0].legend()
axes[1].plot(np.rad2deg(retardances),converted_power);axes[1].set(xlabel='Retardance (deg)',ylabel='Converted fraction',title='Retardance controls conversion')
for ax in axes:ax.grid(alpha=.2)
plt.show()

## ۴. محدودیت مدل کنار سنجه

FWHM و سهم شدت خطی مدل در پنجرهٔ مشخص، سنجهٔ کانون خطی اسکالرند؛ بازده اندازه‌گیری یا تمام‌موج، انرژی محصور دوبعدی یا اثبات فلزنس ساخته‌شده نیستند. پاشندگی فیزیکی، ابیراهی خارج‌محور، تحمل ساخت، نشتی قطبش و بررسی ماکسول قطعهٔ متناهی توسعه می‌خواهند.


In [ ]:
summary=dict(model='1D scalar angular spectrum plus separate analytical Jones matrices',aperture_um=APERTURE_UM,focal_distance_um=FOCAL_DISTANCE_UM,wavelength_um=WAVELENGTH_UM,output_index=OUTPUT_INDEX,dx_um=DX_UM,fft_samples=N_FFT,window_um=DX_UM*N_FFT,focus_window_half_um=FOCUS_WINDOW_HALF_UM,metrics=metrics,refinements=refinements,axial_peak_um=float(z_scan[np.argmax(axial)]),max_jones_error=max_jones_error,library_phase_rms_deg=float(np.rad2deg(np.sqrt(score))),source_sha256=hashlib.sha256((ROOT/'simulations/metasurface_synthesis.py').read_bytes()).hexdigest(),limitations=['Cylindrical scalar focus, not a full-wave metalens','Library aperture uses local periodicity','Jones matrices are ideal analytical elements, not the TE ridge','No physical focusing efficiency is reported'])
(DATA/'wavefront-models.json').write_text(json.dumps(summary,indent=2)+'\n',encoding='utf-8')
print('Saved wavefront-models.json; model checks passed.')

تمرین: فاصلهٔ کانون و عرض دهانه را تغییر و کنترل شبکه و پنجره را تکرار کنید. بیشینهٔ گام فاز و نمونه‌برداری، فاز دقیق و پیرامحوری، و اثر شکل‌دهی دامنه بر اوج و عرض و لوب جانبی با مخرج مشخص را بررسی کنید. تأخیر جونز را تغییر و پوشش فاز را از دامنهٔ تبدیل جدا کنید.

مرجع: [Arbabi و همکاران، کنترل فاز و قطبش](https://doi.org/10.1038/nnano.2015.186). ابرسلول ریج، تمرکز اسکالر و جونز تحلیلی سطح شاهد متفاوت دارند و در گزارش باید مشخص بمانند.
